# Crypto.Net · 10 — NEAR, Aptos, Sui, TON and Cardano

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./10-more-chains.id.ipynb)

One recovery phrase, five more chains. Each section derives the account its wallets show, reads the live network, signs offline and asks a real node to dry-run the result.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Wallet;
using Crypto.Net.Extensions;
using Crypto.Net.Near;
using Crypto.Net.Aptos;
using Crypto.Net.Sui;
using Crypto.Net.Ton;
using Crypto.Net.Cardano;

const string TestPhrase = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about";
var wallet = HdWallet.FromMnemonic(TestPhrase);
new[] { "near-testnet", "aptos-testnet", "sui-testnet", "ton-testnet", "cardano-preprod" }
    .Select(key => ChainCatalog.Get(key))
    .Select(e => { var a = ChainCatalog.DeriveAccount(wallet, e); var r = new { Network = e.Name, Address = a.Address.Value }; (a as IDisposable)?.Dispose(); return r; })

## NEAR

Accounts are implicit (the hex public key) or named (`alice.testnet`). Transactions are Borsh-encoded and signed as ed25519(sha256(bytes)).

In [ ]:
var near = wallet.GetNearAccount(0, NearChain.Testnet);
var nearRpc = new NearRpcClient(NearChain.Testnet);
var meta = await nearRpc.ViewFunctionAsync("wrap.testnet", "ft_metadata");
var nearTx = new NearTransaction
{
    SignerId = near.AccountId, PublicKey = near.PublicKey.ToArray(), Nonce = 1, ReceiverId = "wrap.testnet",
    BlockHash = Crypto.Net.Native.CryptoNative.Base58Decode(await nearRpc.GetLatestBlockHashAsync()),
    Actions = [NearAction.FunctionCall.Json("ft_transfer", new { receiver_id = "bob.testnet", amount = "1" }, deposit: 1)],
};
Console.WriteLine($"wNEAR symbol {meta.GetProperty("symbol")}, block {await nearRpc.GetBlockNumberAsync():N0}");
Console.WriteLine($"signed {nearTx.Sign(near).Length} bytes, hash {nearTx.HashBase58}");

## Aptos

A BCS `RawTransaction` calling `0x1::aptos_account::transfer`. Simulation runs the Move VM without a signature; this key holds no APT, so the VM reports the missing balance, which proves it decoded the transaction.

In [ ]:
var apt = wallet.GetAptosAccount(0, AptosChain.Testnet);
var aptos = new AptosRestClient(AptosChain.Testnet);
var aptTx = await aptos.BuildTransactionAsync(apt.Address.Value, AptosEntryFunction.Transfer("0x1", 1));
var sim = await aptos.SimulateAsync(aptTx, apt.PublicKey);
Console.WriteLine($"ledger {(await aptos.GetLedgerInfoAsync()).BlockHeight:N0}, sequence {aptTx.SequenceNumber}");
Console.WriteLine($"simulation: {sim.VmStatus}, gas {sim.GasUsed}");
Console.WriteLine($"signed hash {AptosRawTransaction.HashOf(aptTx.Sign(apt))}");

## Sui

A programmable transaction splits the gas coin and transfers the piece. To dry-run with real coins we borrow a funded address from the chain; nothing is signed for it.

In [ ]:
var sui = wallet.GetSuiAccount(0, SuiChain.Testnet);
var suiClient = new SuiGraphQlClient(SuiChain.Testnet);
var found = await suiClient.QueryAsync("{ objects(first: 1, filter: { type: \"0x2::coin::Coin<0x2::sui::SUI>\" }) { nodes { owner { ... on AddressOwner { address { address } } } } } }");
string funded = found.GetProperty("objects").GetProperty("nodes")[0].GetProperty("owner").GetProperty("address").GetProperty("address").GetString()!;
var data = await suiClient.BuildAsync(new SuiTransactionBuilder(funded).TransferSui(sui.Address.Value, 1));
var effects = await suiClient.SimulateAsync(data.Serialize());
Console.WriteLine($"dry run {(effects.Success ? "succeeded" : effects.Error)}, gas {effects.NetGas:N0} MIST");
Console.WriteLine($"node digest {effects.Digest} = ours {data.Digest}: {effects.Digest == data.Digest}");

## TON

TON wallets are contracts: the address is the hash of the wallet's code and data. W5 is the default; the first transfer deploys it. TON Center emulates the wallet code on our signed body to price it.

In [ ]:
var ton = wallet.GetTonAccount(0, TonChain.Testnet);
var center = new TonCenterClient(TonChain.Testnet);
var state = await center.GetAccountStateAsync(ton.Address.Value);
uint seqno = await center.GetSeqnoAsync(ton.Address.Value);
var body = ton.CreateTransferBody(seqno, [TonMessage.WithComment(ton.WalletAddress, 1, "Crypto.Net notebook")]);
var fees = await center.EstimateFeeAsync(ton, body, includeStateInit: !state.IsActive);
Console.WriteLine($"{ton.Address.Value} is {state.State}, seqno {seqno}");
Console.WriteLine($"estimated fees {TonUnits.Nano(fees.Total)} TON; message hash {Convert.ToHexStringLower(TonMessage.NormalizedHash(ton.CreateTransfer(seqno, [])))}");

## Cardano

Cardano keys come from the mnemonic entropy (Icarus) through BIP32-Ed25519. The test phrase holds test ADA on preprod, so we can select real UTxOs, return change and pay the exact fee, then stop before submitting.

In [ ]:
var ada = wallet.GetCardanoAccount(0, 0, CardanoChain.Preprod);
var koios = new KoiosClient(CardanoChain.Preprod);
Console.WriteLine($"balance {await koios.GetBalanceAsync(ada.Address)} tADA");
try
{
    var adaTx = await koios.BuildTransferAsync(ada, [new CardanoOutput(ada.EnterpriseAddress, 2_000_000)], message: "Crypto.Net notebook");
    byte[] signedAda = adaTx.Sign(ada);
    Console.WriteLine($"{adaTx.Inputs.Count} input(s), {adaTx.Outputs.Count} outputs, fee {adaTx.Fee:N0} lovelace, {signedAda.Length} bytes");
    Console.WriteLine($"tx id {adaTx.Id}");
}
catch (InvalidOperationException e) { Console.WriteLine(e.Message); }

In [ ]:
near.Dispose(); apt.Dispose(); sui.Dispose(); ton.Dispose(); ada.Dispose();
nearRpc.Dispose(); aptos.Dispose(); suiClient.Dispose(); center.Dispose(); koios.Dispose(); wallet.Dispose();
"disposed"